In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Load the cleaned dataset (isFlaggedFraud already removed)
df = pd.read_csv('paysim_balanced_clean.csv')
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)

# Filter high-risk transaction types (TRANSFER & CASH_OUT)
df_filtered = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]

# Features — isFlaggedFraud is no longer in the dataset
features = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
X = df_filtered[features]
y = df_filtered['isFraud']

# Split: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Preprocessing Complete: Data filtered and scaled.')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_preds = nb_model.predict(X_test_scaled)

print('Models Trained: Logistic Regression and Naive Bayes are ready.')

In [ ]:
def show_evaluation(y_true, y_pred, model_name):
    print(f"\n{'='*20} {model_name} {'='*20}")
    print(f"Accuracy Score: {accuracy_score(y_true, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Mule'],
                yticklabels=['Normal', 'Mule'])
    plt.title(f'Confusion Matrix: {model_name}')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.show()

show_evaluation(y_test, lr_preds, 'Logistic Regression')
show_evaluation(y_test, nb_preds, 'Naive Bayes')

In [ ]:
import joblib, json

joblib.dump(lr_model, 'lr_model.pkl')
joblib.dump(nb_model, 'nb_model.pkl')
joblib.dump(scaler,   'scaler.pkl')
json.dump(features, open('feature_columns.json', 'w'))
print('Saved: lr_model.pkl, nb_model.pkl, scaler.pkl, feature_columns.json')